# 🛡️ WikiGuard AI — Challenge 4

An evidence-first fact-consistency checker for Wikimedia Structured Contents.

**Rule:** Possible mismatch detected — human verification recommended.


In [1]:
!pip -q install kagglehub[pandas-datasets] pandas numpy scikit-learn pyarrow


In [2]:
import pandas as pd
import numpy as np
from kagglehub import KaggleDatasetAdapter
from wikiguard_core import parse_any, flatten_pairs, compare_article, summarize_results, build_article_text

DATASET = 'wikimedia-foundation/wikipedia-structured-contents'
PARQUET = 'enwiki/data/enwiki_namespace_0_00008.parquet'

df = pd.DataFrame()
try:
    df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, DATASET, PARQUET)
    print('✅ Wikimedia dataset loaded successfully')
    print('Rows:', len(df))
    print('Columns:', df.columns.tolist())
except Exception as error:
    print('Dataset loading error:', error)
    print('The comparison engine can still be tested with the deterministic demonstration below.')


In [3]:
# Search the real Wikimedia shard when it is available.
if not df.empty:
    data = df.copy()
    for column in ['name','abstract','description','sections','infoboxes']:
        if column in data.columns:
            data[column] = data[column].fillna('').astype(str)
    data = data[data['name'].str.strip() != ''].reset_index(drop=True)

    query = input('🔎 Enter article/topic: ').strip().lower()
    matches = data[data['name'].str.lower().str.contains(query, regex=False, na=False)].head(10)
    print('\nMatching articles:')
    for i, title in enumerate(matches['name'], 1):
        print(f'{i}. {title}')
else:
    print('Live search skipped because the dataset was not loaded in this runtime.')


In [4]:
# Fact Detective scan for a selected live article.
if not df.empty and 'matches' in globals() and not matches.empty:
    choice = input(f'Select article number (1-{len(matches)}): ').strip()
    if choice.isdigit() and 1 <= int(choice) <= len(matches):
        selected = matches.iloc[int(choice)-1]
        print('='*72)
        print('🕵️ FACT DETECTIVE MODE')
        print('='*72)
        print('Article:', selected['name'])
        results = compare_article(selected)
        summary = summarize_results(results)
        print('Facts analyzed:', summary['facts_analyzed'])
        for n, result in enumerate(results, 1):
            icon = '✅' if result['status']=='CONSISTENT' else '⚠️'
            print(f'\n{icon} FACT {n}: {result["fact"]}')
            print('Structured:', result['structured'])
            print('Article:', result['article_value'])
            print('Status:', result['status'])
            if result['evidence']:
                print('Evidence:', result['evidence'])
        print('\nRecommendation: Human verification recommended.')


## 🧪 Deterministic demonstration

This test record is intentionally synthetic. It proves the comparison engine can detect a possible date mismatch without pretending that the example is a live Wikimedia claim.


In [5]:
demo = pd.Series({
    'name':'Demo Person',
    'description':'A demonstration record for testing WikiGuard AI.',
    'abstract':'Demo Person was born on 14 March 1988 in Example City. The person is used only for a reproducible software test.',
    'sections':'Early life: Demo Person was born on 14 March 1988. Career information is omitted from this test.',
    'infoboxes': {'birth_date':'12 March 1988','occupation':'Researcher','country':'United Kingdom'}
})

print('='*72)
print('🛡️ WIKIGUARD AI — FACT INVESTIGATION')
print('='*72)
results = compare_article(demo)
for n, result in enumerate(results, 1):
    print(f'\nFACT {n}')
    print('Structured value :', result['structured'])
    print('Article evidence :', result['article_value'])
    print('Status            :', result['status'])
    print('Evidence          :', result['evidence'])

summary = summarize_results(results)
print('\n'+'-'*72)
print('FINAL INVESTIGATION REPORT')
print('-'*72)
print('Facts analyzed     :', summary['facts_analyzed'])
print('Consistent         :', summary['consistent'])
print('Possible mismatch  :', summary['possible_mismatch'])
print('Not found          :', summary['not_found'])
print('\n⚠️ Possible mismatch detected — human verification recommended.')


## ✅ Challenge 4 coverage

- Structured fact extraction
- Article-text evidence search
- Normalization
- Possible mismatch detection
- Evidence display
- Human-verification workflow
- Final investigation summary


In [6]:
print('WikiGuard AI — Challenge 4 COMPLETE')
print('Evidence-first fact consistency prototype ready.')
